# 🛡️ VulHunter — Semantic Branch (Qwen2.5-Coder-1.5B-Instruct)

> **Notebook huấn luyện nhánh Semantic Only** trên Kaggle 1×P100 16GB.

| Điều kiện | Cấu hình |
|---|---|
| GPU | **1×P100 16GB** |
| Internet | ON — pull tokenizer/model từ HuggingFace |
| Backbone | `Qwen2.5-Coder-1.5B-Instruct` (full fine-tune, ~6GB VRAM) |
| Tasks | 3 tasks: Binary Detection · CWE Classification · Severity Prediction |
| Data | `train/validation/test.jsonl` đã pre-tokenized → mount read-only `/kaggle/input/...` |

**Chuẩn bị trước khi Run:**
1. Local đã chạy `python notebooks/prepare_kaggle_dataset.py` và upload lên Kaggle Dataset `vulhunter-pre-tokenized` → **Add Input**.
2. Notebook Settings: **Accelerator = GPU P100**, **Internet = ON**.

---
# Part 1: Setup Environment
---

## 1.1 Định vị project & kiểm tra GPU

Nếu code chưa có trong `/kaggle/working/VulHunter`, cell này tự `git clone`.

In [ ]:
import sys
from pathlib import Path
import subprocess

CANDIDATES = [Path("/kaggle/working/VulHunter"), Path("/kaggle/input/VulHunter-code"), Path("/kaggle/input/vulhunter-code"), Path.cwd(), Path.cwd().parent]
ROOT = next((p for p in CANDIDATES if (p / "pyproject.toml").exists()), None)
if ROOT is None:
    print("[INFO] Chưa có repo — git clone ...")
    subprocess.run(["git", "clone", "https://github.com/NhatWoan-20/VulHunter", "/kaggle/working/VulHunter"], check=True)
    ROOT = Path("/kaggle/working/VulHunter")
elif (ROOT / ".git").exists():
    print(f"[INFO] Repo đã có tại {ROOT} — đồng bộ mới nhất từ GitHub...")
    subprocess.run(["git", "-C", str(ROOT), "fetch", "origin", "main"], check=False)
    subprocess.run(["git", "-C", str(ROOT), "reset", "--hard", "origin/main"], check=False)
print(f"ROOT = {ROOT}")
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "notebooks"))

from kaggle_utils import print_gpu_info
print_gpu_info()


## 1.2 Cài dependencies (~3 phút, có cache)

Kaggle image đã có `torch` nhưng thiếu `transformers`, `sentencepiece`, `scikit-learn`. Cell idempotent — chạy lại không sao.

In [ ]:
import subprocess, sys

try:
    get_ipython().run_line_magic("pip", f"install -q -r {ROOT / 'requirements.txt'}")
except NameError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)

try:
    get_ipython().run_line_magic("pip", "install -q accelerate matplotlib seaborn tqdm pyyaml")
except NameError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "accelerate", "matplotlib", "seaborn", "tqdm", "pyyaml"], check=True)

import torch, transformers, sklearn, yaml
print(f"torch {torch.__version__} (CUDA {torch.version.cuda}) | transformers {transformers.__version__} | sklearn {sklearn.__version__}")
print(f"GPUs: {torch.cuda.device_count()} | AMP fp16: {'OK' if torch.cuda.is_available() else 'no GPU'}")


## 1.3 Setup — tự detect data read-only & tạo thư mục output

Không copy data — chỉ tạo `hf_cache`, `checkpoints`, `outputs` trong `/kaggle/working` (writable). Data đọc thẳng từ `/kaggle/input/vulhunter-pre-tokenized/`.

In [ ]:
from kaggle_utils import setup_kaggle_env, get_data_root, get_checkpoint_dir, get_working_root
setup_kaggle_env()
print(f"\nCheckpoint : {get_checkpoint_dir()}")
print(f"Working    : {get_working_root()}")
print(f"Data root  : {get_data_root()}")


## 1.4 Kiểm tra dataset mount & data READY?

Nếu `MISSING` → **Add Input** dataset `vulhunter-pre-tokenized` (bên phải Kaggle UI).  
Nếu `✅ READY` → **sang training ngay**.

In [ ]:
from pathlib import Path
import json
from kaggle_utils import get_data_root, inspect_splits, print_inspect
data_root = get_data_root()
print("Scan /kaggle/input:")
found = list(Path("/kaggle/input").rglob("*.jsonl"))
for p in found[:10]:
    print(f"  {p}  {p.stat().st_size/1e6:.1f} MB")
if not found:
    print("  (chưa thấy .jsonl — kiểm tra Add Input)")
print()
info = inspect_splits(data_root)
print_inspect(info)
if found and (data_root / "train.jsonl").exists():
    s = json.loads(open(data_root / "train.jsonl", encoding="utf-8").readline())
    print(f"\nSample keys: {list(s.keys())[:15]}")
    if info["ready_for_training"]:
        print("\n✅ READY — data pre-tokenized. Sang training ngay!")
    else:
        print("\n⚠️ Cần tokenize — chạy preprocessing.")


---
# Part 2: Training
---

## 2.1 Cấu hình Training

| Backbone | VRAM | Batch size | Grad Accum | Eff batch | Thời gian |
|---|---|---|---|---|---|
| `Qwen2.5-Coder-1.5B-Instruct` | ~6GB / 16GB P100 | 2 | 4 | 8 | ~1.5-2h |

Config: `configs/kaggle/train_kaggle.yaml` + `configs/kaggle/model_kaggle.yaml`

In [ ]:
import sys
from pathlib import Path
for p in [Path("/kaggle/working/VulHunter"), Path.cwd(), Path.cwd().parent]:
    if (p / "pyproject.toml").exists():
        ROOT = p; break
else:
    ROOT = Path("/kaggle/working/VulHunter") if Path("/kaggle/working/VulHunter").exists() else Path.cwd()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "notebooks"))
from kaggle_utils import setup_kaggle_env, get_data_root, get_checkpoint_dir, get_working_root, inspect_splits, print_inspect
setup_kaggle_env()

MODE = "semantic_only"

# Config cho 1.5B full fine-tune trên P100
train_cfg = str(ROOT / "configs/kaggle/train_kaggle.yaml")
model_cfg = str(ROOT / "configs/kaggle/model_kaggle.yaml")

# Tùy chọn override (để None = dùng config):
EPOCHS_OVERRIDE = None       # ví dụ 1 để smoke test
MAX_LEN_OVERRIDE = None      # ví dụ 1024 để tiết kiệm VRAM

print(f"train_cfg : {train_cfg}")
print(f"model_cfg : {model_cfg}")
print(f"mode      : {MODE}")

# Data read-only
data_root = get_data_root()
train_data = str(data_root / "train.jsonl")
val_data   = str(data_root / "validation.jsonl")
print(f"train_data: {train_data}  (read-only OK)")
print(f"val_data  : {val_data}  (read-only OK)")

import yaml, torch
tc = yaml.safe_load(open(train_cfg, encoding="utf-8"))["training"]
mc = yaml.safe_load(open(model_cfg, encoding="utf-8"))["model"]
n_gpu = torch.cuda.device_count() if torch.cuda.is_available() else 1
eff = tc["batch_size"] * tc["gradient_accumulation_steps"] * n_gpu
print(f"\nBackbone      : {mc['semantic']['backbone']}")
print(f"Per-device bs : {tc['batch_size']}  accum={tc['gradient_accumulation_steps']}  GPUs={n_gpu}  => eff batch = {eff}")
print(f"LR            : {tc['learning_rate']}")
print(f"Epochs        : {tc['epochs']}  patience={tc['early_stopping']['patience']}  AMP={'ON' if tc.get('use_amp') else 'OFF'}")


## 2.2 Chạy training

Checkpoint lưu vào `/kaggle/working/models/checkpoints/best.pt` — **nhớ Save Version** trước khi hết session.

In [ ]:
import subprocess, sys, torch
from pathlib import Path
from kaggle_utils import get_checkpoint_dir

ckpt_dir = str(get_checkpoint_dir())

# Single GPU — chạy trực tiếp
launcher = [sys.executable, "-u"]
extra_args = ["--batch-size", str(tc["batch_size"]), "--grad-accum", str(tc["gradient_accumulation_steps"])]
print(f"ℹ️ Training trên 1×P100 (eff batch = {eff}).")

# Tự động Resume nếu đã upload checkpoint từ phiên trước
from kaggle_utils import find_resume_checkpoint
auto_resume = find_resume_checkpoint()
if auto_resume:
    extra_args += ["--resume", str(auto_resume)]
    print(f"🔄 PHÁT HIỆN CHECKPOINT: {auto_resume} -> Tự động RESUME!")

cmd = launcher + [
    "scripts/training/train.py",
    "--mode", MODE,
    "--config", train_cfg,
    "--model-config", model_cfg,
    "--train-data", train_data,
    "--val-data", val_data,
    "--checkpoint-dir", ckpt_dir
] + extra_args

if EPOCHS_OVERRIDE:  cmd += ["--epochs", str(EPOCHS_OVERRIDE)]
if MAX_LEN_OVERRIDE: cmd += ["--max-length", str(MAX_LEN_OVERRIDE)]

print("CMD:", " ".join(cmd))
print(f"\nBắt đầu training ({MODE})...\n")

p = subprocess.Popen(
    cmd, cwd=str(ROOT),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, universal_newlines=True
)
for line in iter(p.stdout.readline, ""):
    print(line, end="", flush=True)
p.stdout.close()
returncode = p.wait()

print(f"\nExit code: {returncode}")
if returncode == 0:
    from kaggle_utils import save_kaggle_output_checkpoint
    save_kaggle_output_checkpoint(Path(ckpt_dir) / "best.pt")
    print("✅ Training xong — sang evaluation")
else:
    print(f"❌ Exit code: {returncode}")
    for lf in [Path(ckpt_dir).parent.parent/"runs"/"train.log", Path("/kaggle/working/runs/train.log"), ROOT/"runs"/"train.log"]:
        if lf.exists():
            print(f"\n--- [TRAIN.LOG TAIL] ---")
            print("\n".join(lf.read_text(encoding="utf-8", errors="ignore").splitlines()[-30:]))
            break


## 2.3 Kiểm tra kết quả & vẽ biểu đồ training

Checkpoint `best.pt` được chọn theo `val F1 binary` (primary metric).

In [ ]:
import json
from pathlib import Path
from kaggle_utils import get_checkpoint_dir, get_working_root
ckpt = get_checkpoint_dir() / "best.pt"
hist = get_checkpoint_dir() / "training_history.json"
print(f"ckpt : {ckpt.exists()}  {ckpt.stat().st_size/1e6:.1f} MB" if ckpt.exists() else f"ckpt MISSING: {ckpt}")
print(f"hist : {hist.exists()}  {hist.stat().st_size/1e3:.0f} KB" if hist.exists() else f"hist MISSING: {hist}")
if hist.exists():
    h = json.loads(hist.read_text(encoding="utf-8"))
    print(f"  epochs: {len(h)}  best val F1: {max(e['val_metrics'].get('binary',{}).get('f1',0) for e in h):.4f}")
    for e in h[-4:]:
        print(f"  epoch {e['epoch']:2d}  train={e['train_loss'].get('total',0):.4f}  val_f1={e['val_metrics'].get('binary',{}).get('f1',0):.4f}  auc={e['val_metrics'].get('binary',{}).get('auc',0):.4f}")
    try:
        import matplotlib.pyplot as plt
        epochs = [e['epoch'] for e in h]
        val_f1 = [e['val_metrics'].get('binary',{}).get('f1',0) for e in h]
        plt.figure(figsize=(8,4))
        plt.plot(epochs, val_f1, marker='o', label='val binary F1')
        plt.xlabel('epoch'); plt.ylabel('F1'); plt.legend(); plt.grid(True, alpha=0.3)
        plt.title('Val F1 per epoch (Semantic Only — Qwen 1.5B · P100)')
        plt.savefig(str(get_working_root() / "val_f1_plot.png"), dpi=150, bbox_inches="tight")
        plt.show()
    except Exception as ex:
        print(f"plot skip: {ex}")
print("\n✅ Xong training — sang evaluation")


---
# Part 3: Evaluation
---

## 3.1 Chạy evaluate

Đánh giá 3 Tasks (Binary · CWE · Severity) trên tập `test.jsonl` (read-only). Chạy ~2 phút trên P100.

In [ ]:
import sys
from pathlib import Path
for p in [Path("/kaggle/working/VulHunter"), Path.cwd(), Path.cwd().parent]:
    if (p / "pyproject.toml").exists():
        ROOT = p; break
else:
    ROOT = Path("/kaggle/working/VulHunter") if Path("/kaggle/working/VulHunter").exists() else Path.cwd()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "notebooks"))
from kaggle_utils import setup_kaggle_env, get_data_root, get_checkpoint_dir, get_working_root
setup_kaggle_env()

CHECKPOINT = str(get_checkpoint_dir() / "best.pt")
OUTPUT     = str(get_working_root() / "outputs/metrics/evaluation_report.json")
BATCH_EVAL = 16

import subprocess
test_data = str(get_data_root() / "test.jsonl")
cmd = [sys.executable, "scripts/evaluation/evaluate.py",
       "--checkpoint", CHECKPOINT, "--test-data", test_data, "--output", OUTPUT, "--batch-size", str(BATCH_EVAL)]
print("CMD:", " ".join(cmd))
result = subprocess.run(cmd, cwd=str(ROOT))
print(f"\nexit code: {result.returncode}")
if Path(OUTPUT).exists():
    print(f"Report: {OUTPUT}  {Path(OUTPUT).stat().st_size/1e3:.1f} KB")
else:
    print("[WARN] report chưa tạo — kiểm tra CHECKPOINT path.")

## 3.2 Tổng quan metrics — 3 Tasks (Binary, CWE, Severity)

In [ ]:
import json
from pathlib import Path
from kaggle_utils import get_working_root
OUTPUT = str(get_working_root() / "outputs/metrics/evaluation_report.json")
if not Path(OUTPUT).exists():
    print(f"Không thấy {OUTPUT}")
else:
    r = json.loads(Path(OUTPUT).read_text(encoding="utf-8"))
    print(f"samples={r.get('num_samples')}  checkpoint={Path(r.get('checkpoint','')).name}  mode={r.get('config',{}).get('mode','?')}")
    print()
    for task in ["binary","cwe","severity"]:
        m = r.get("metrics",{}).get(task)
        if not m:
            print(f"[{task:15s}] — no metrics")
        else:
            print(f"[{task:15s}] F1={m.get('f1',0):.4f}  P={m.get('precision',0):.4f}  R={m.get('recall',0):.4f}  Acc={m.get('accuracy',0):.4f}  AUC={m.get('auc',0):.4f}")
            if "per_class" in m:
                for cls, v in m["per_class"].items():
                    print(f"    {cls:20s} F1={v.get('f1',0):.3f}  P={v.get('precision',0):.3f}  R={v.get('recall',0):.3f}")
    print("\n(Hint: binary F1 là primary metric — early stopping & best.pt chọn theo nó)")

## 3.3 Biểu đồ

In [ ]:
import json
from pathlib import Path
from kaggle_utils import get_working_root
OUTPUT = str(get_working_root() / "outputs/metrics/evaluation_report.json")
try:
    import matplotlib.pyplot as plt
    r = json.loads(Path(OUTPUT).read_text(encoding="utf-8"))
    m = r.get("metrics",{})
    tasks = ["binary","cwe","severity"]
    f1s = [m.get(t,{}).get("f1",0) for t in tasks]
    plt.figure(figsize=(7,4))
    bars = plt.bar(tasks, f1s, color=["#4C78A8","#F58518","#54A24B"])
    plt.ylim(0,1); plt.ylabel("F1"); plt.title("VulHunter — F1 per task (test)")
    for b,v in zip(bars,f1s):
        plt.text(b.get_x()+b.get_width()/2, b.get_height()+0.02, f"{v:.3f}", ha="center", fontsize=9)
    plt.grid(axis="y", alpha=0.3)
    plt.savefig(str(get_working_root()/"f1_per_task.png"), dpi=150, bbox_inches="tight")
    plt.show()
    cwe = m.get("cwe",{}).get("per_class",{})
    if cwe:
        names=list(cwe.keys()); vals=[cwe[k].get("f1",0) for k in names]
        plt.figure(figsize=(9,4)); plt.bar(names, vals, color="#72B7B2")
        plt.xticks(rotation=30, ha="right"); plt.ylim(0,1); plt.ylabel("F1"); plt.title("CWE per-class F1")
        plt.grid(axis="y", alpha=0.3); plt.tight_layout()
        plt.savefig(str(get_working_root()/"cwe_per_class.png"), dpi=150, bbox_inches="tight")
        plt.show()
except Exception as e:
    print(f"plot error: {e}")
    import traceback; traceback.print_exc()

In [ ]:
import json
from pathlib import Path
from kaggle_utils import get_checkpoint_dir, get_working_root
hist = get_checkpoint_dir() / "training_history.json"
if not hist.exists():
    print(f"Không có {hist} — bỏ qua (chỉ có khi train trong cùng session)")
else:
    import matplotlib.pyplot as plt
    h = json.loads(hist.read_text(encoding="utf-8"))
    epochs=[e["epoch"] for e in h]
    tr=[e["train_loss"].get("total",0) for e in h]
    va=[e["val_loss"].get("total",0) for e in h]
    f1=[e["val_metrics"].get("binary",{}).get("f1",0) for e in h]
    fig, ax1 = plt.subplots(figsize=(8,4))
    ax1.plot(epochs,tr,marker="o",label="train loss",color="#4C78A8")
    ax1.plot(epochs,va,marker="s",label="val loss",color="#F58518")
    ax1.set_xlabel("epoch"); ax1.set_ylabel("loss")
    ax2=ax1.twinx()
    ax2.plot(epochs,f1,marker="^",label="val F1",color="#54A24B",linestyle="--")
    ax2.set_ylabel("val F1")
    l1,lb1=ax1.get_legend_handles_labels(); l2,lb2=ax2.get_legend_handles_labels()
    ax1.legend(l1+l2, lb1+lb2, loc="best")
    plt.title("Training history")
    plt.grid(alpha=0.3)
    plt.savefig(str(get_working_root()/"training_history.png"), dpi=150, bbox_inches="tight")
    plt.show()
print("\n✅ Xong evaluation")

---
# Part 4: MLOps Deployment
---

# 🚀 MLOps: FastAPI Deployment

> Triển khai mô hình dưới dạng REST API Server để dự đoán lỗi bảo mật trực tiếp thông qua HTTP request.

In [ ]:
!uvicorn scripts.api_deployment:app --host 0.0.0.0 --port 8000
